# 02 · EDA cuantitativo — textometría y estilo

**Fase 2 del plan.** Pregunta rectora: *¿qué información del tweet permite inferir el rango
de edad?* Aquí se mide, no se asume: cada familia de features se somete a prueba con tamaño
de efecto y se documenta con el formato OBSERVACIÓN → INTERPRETACIÓN → HIPÓTESIS →
CONSECUENCIA.

**Reglas.** Todo se calcula **solo sobre train**. Las features densas de este notebook son
deterministas y locales al documento (no aprenden parámetros del corpus), por lo que pueden
calcularse también sobre test sin leakage; lo que nunca se hace con test es *seleccionar*
features ni ajustar umbrales.

**Salidas.** `features/dense_train.parquet`, `features/dense_test.parquet`,
`reports/tables/eda_kruskal.csv`, `eda_mutual_information.csv`, `eda_binarias.csv`,
`eda_estilo_por_clase.csv` y figuras `reports/figures/eda_*.png`.

## 0. Configuración y cálculo de features densas

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg, data, eda, textometry
from src.config import set_seed

set_seed()
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

train = data.load_train()
test = data.load_test()
y_train = train[cfg.TARGET].astype(str)

FORCE_RECOMPUTE = False
cache_train = cfg.FEATURES / "dense_train.parquet"
cache_test = cfg.FEATURES / "dense_test.parquet"
if cache_train.exists() and cache_test.exists() and not FORCE_RECOMPUTE:
    dense_train = pd.read_parquet(cache_train)
    dense_test = pd.read_parquet(cache_test)
    print("features densas cargadas de caché")
else:
    dense_train = textometry.extract_textometry(train)
    dense_test = textometry.extract_textometry(test)
    dense_train.to_parquet(cache_train, index=False)
    dense_test.to_parquet(cache_test, index=False)
    print("features densas recalculadas y cacheadas")

features = textometry.TEXTOMETRY_FEATURES + textometry.STYLE_FEATURES
print(f"features densas: {len(features)} | train {dense_train.shape} | test {dense_test.shape}")
dense_train.head(3)

features densas recalculadas y cacheadas
features densas: 38 | train (169336, 38) | test (112892, 38)


,n_chars,n_words,n_unique_words,ttr,cttr,hapax_ratio,lexical_entropy,avg_word_len,max_word_len,std_word_len,n_sentences,avg_sentence_len_words,stopword_ratio,content_ratio,long_word_ratio,upper_ratio,digit_ratio,punct_ratio,n_excl,n_quest,n_open_punct,multi_punct_count,elongated_count,all_caps_count,has_all_caps,n_emoji,emoji_diversity,has_emoji,n_emoticon,has_emoticon,n_hashtag,n_mention,n_url,is_rt,starts_mention,n_repeated_char_runs,punct_per_word,unique_char_ratio
0,131,24,23,0.958333,3.319764,0.916667,4.501629,4.500000,12,3.109126,1,24.000000,0.500000,0.500000,0.208333,0.028037,0.000000,0.007634,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0.041667,0.190840
1,22,1,1,1.000000,0.707107,1.000000,-0.000000,22.000000,22,0.000000,2,0.500000,0.000000,1.000000,1.000000,0.266667,0.090909,0.227273,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,5.000000,0.636364
2,139,17,17,1.000000,2.915476,1.000000,4.087463,7.235294,23,5.599679,3,5.666667,0.294118,0.705882,0.411765,0.064220,0.043165,0.057554,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,3,0.470588,0.237410


## 1. Textometría por clase

**OBSERVACIÓN (evidencia).** La longitud del tweet y la riqueza léxica crecen de forma
monótona con el rango de edad; las diferencias no son ruido: se ven en medias y medianas.

**INTERPRETACIÓN.** Es el patrón clásico de *author profiling*: los usuarios mayores escriben
tweets más largos y más elaborados, con menor densidad de marcas informales.

**HIPÓTESIS H1.** Las medidas textométricas (longitud, riqueza léxica, longitud media de
palabra) aportan señal directa y deberían estar entre las features densas más informativas.

**CONSECUENCIA.** Se incluyen como familia `GROUP_D_TEXTOMETRY` y se medirá su aporte
incremental en la ablación de la Fase 18.

In [2]:
summary = textometry.textometry_summary(dense_train, y_train)
summary.round(3)

_target,13-17,18-24,25-34,35-49,50-64,65-xx
n_chars,73.475,72.565,84.771,96.027,104.406,110.966
n_words,10.981,11.111,12.570,13.657,14.540,15.398
n_unique_words,10.398,10.499,11.909,12.931,13.717,14.566
ttr,0.964,0.963,0.963,0.962,0.957,0.958
cttr,2.122,2.135,2.285,2.390,2.463,2.555
hapax_ratio,0.935,0.933,0.931,0.928,0.920,0.921
lexical_entropy,3.038,3.063,3.268,3.406,3.496,3.619
avg_word_len,6.729,6.529,6.713,7.193,7.412,7.313
max_word_len,15.837,15.190,16.391,17.720,17.985,18.435
std_word_len,4.266,4.011,4.275,4.507,4.485,4.578


## 2. Kruskal-Wallis con tamaño de efecto (ε²)

In [3]:
kruskal_table = eda.kruskal_tests(dense_train, y_train, features)
kruskal_table.to_csv(cfg.TABLES / "eda_kruskal.csv", index=False)
print(eda.fig_feature_heatmap(kruskal_table, dense_train, y_train, top_n=20))
kruskal_table.head(15).round(4)

/home/carlos/venv/lib/python3.14/site-packages/scipy/stats/_stats_py.py:8492: RuntimeWarning: invalid value encountered in scalar divide
  h /= ties


/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/eda_heatmap_features.png


,feature,H,p_value,epsilon_sq,mean_13_17,mean_65_xx
0,n_chars,21257.5948,0.0,0.1255,73.4746,110.9659
1,unique_char_ratio,11332.0554,0.0,0.0669,0.4233,0.3292
2,cttr,11251.8700,0.0,0.0664,2.1216,2.5555
3,lexical_entropy,11038.1738,0.0,0.0652,3.0382,3.6185
4,n_unique_words,10921.2803,0.0,0.0645,10.3977,14.5663
5,n_words,10256.6029,0.0,0.0605,10.9810,15.3985
6,n_mention,8829.8762,0.0,0.0521,0.7518,1.3236
7,max_word_len,7938.1726,0.0,0.0469,15.8371,18.4348
8,long_word_ratio,6768.3923,0.0,0.0399,0.2870,0.3620
9,n_sentences,6726.1601,0.0,0.0397,1.7495,2.2247


**INTERPRETACIÓN.** Con n = 169.336 casi cualquier diferencia es "significativa" (p < 0,05),
por lo que el criterio de decisión es el **tamaño de efecto ε²**, no el valor p. Las
features con ε² más alto son las que concentran la señal de clase.

## 3. Features binarias: χ² y V de Cramér

In [4]:
binary_features = ["has_emoji", "has_emoticon", "is_rt", "starts_mention", "has_all_caps"]
binary_table = eda.chi2_binary_tests(dense_train, y_train, binary_features)
binary_table.to_csv(cfg.TABLES / "eda_binarias.csv", index=False)
binary_table.round(4)

,feature,chi2,p_value,cramers_v
0,has_all_caps,5304.7521,0.0,0.1770
1,is_rt,4532.1092,0.0,0.1636
2,starts_mention,3028.6065,0.0,0.1337
3,has_emoticon,2242.1404,0.0,0.1151


## 3b. Features constantes y alfabeto del corpus

In [5]:
constant_table = textometry.constant_feature_report(dense_train, features)
constant_table.to_csv(cfg.TABLES / "eda_constantes.csv", index=False)
print("features constantes:", constant_table.loc[constant_table["es_constante"], "feature"].tolist())

import collections
characters = collections.Counter(ch for value in train[cfg.TEXT_COL] for ch in value)
non_ascii = {ch: n for ch, n in characters.items() if ord(ch) > 127}
print(f"caracteres distintos en el corpus: {len(characters)} | no-ASCII: {len(non_ascii)}")
print("no-ASCII:", [(f"U+{ord(ch):04X}", n) for ch, n in non_ascii.items()])
print("total de emojis en train:", int(dense_train["n_emoji"].sum()),
      "| tweets con emoji:", int((dense_train["n_emoji"] > 0).sum()))
print("tildes (á/é/í/ó/ú/ñ):",
      {ch: characters.get(ch, 0) for ch in "áéíóúñÁÉÍÓÚÑ"})
constant_table.head(10)

features constantes: ['n_emoji', 'emoji_diversity', 'has_emoji']


caracteres distintos en el corpus: 96 | no-ASCII: 1
no-ASCII: [('U+00BF', 51)]
total de emojis en train: 0 | tweets con emoji: 0
tildes (á/é/í/ó/ú/ñ): {'á': 0, 'é': 0, 'í': 0, 'ó': 0, 'ú': 0, 'ñ': 0, 'Á': 0, 'É': 0, 'Í': 0, 'Ó': 0, 'Ú': 0, 'Ñ': 0}


,feature,n_unique,top_value,top_share,varianza,es_constante
0,n_emoji,1,0.0,1.0000,0.000000,True
1,emoji_diversity,1,0.0,1.0000,0.000000,True
2,has_emoji,1,0.0,1.0000,0.000000,True
3,has_all_caps,2,0.0,0.5364,0.248674,False
4,starts_mention,2,0.0,0.7994,0.160333,False
5,is_rt,2,0.0,0.6363,0.231436,False
6,n_open_punct,2,0.0,0.9997,0.000301,False
7,has_emoticon,2,0.0,0.9748,0.024603,False
8,n_url,6,0.0,0.5951,0.313935,False
9,elongated_count,8,0.0,0.9537,0.066995,False


**OBSERVACIÓN.** El corpus usa **96 caracteres distintos y un solo carácter no-ASCII**
(`¿`, 88 apariciones): no hay tildes, ni `ñ`, ni emojis, ni puntuación tipográfica. Las
features de emoji son constantes (todas cero).

**INTERPRETACIÓN.** El texto fue **transliterado a ASCII** por el proceso que generó el
corpus (típico de exportaciones antiguas o de pipelines de normalización). Es un artefacto
del corpus, no una propiedad de los autores.

**CONSECUENCIAS.** (1) las features de emoji se excluyen del modelado por varianza nula;
(2) las variantes de normalización de tildes/emojis de la Fase 4 son no-ops y se documentan
como tales; (3) el alfabeto mínimo (96 caracteres) hace que los n-gramas de carácter sean
baratos y potencialmente muy eficaces, lo que refuerza la hipótesis H6.

## 4. Información mutua con la clase

In [6]:
mi_table = eda.mutual_information_table(dense_train, y_train, features)
mi_table.to_csv(cfg.TABLES / "eda_mutual_information.csv", index=False)
print(eda.fig_mi_ranking(mi_table, top_n=20))
mi_table.head(15).round(5)

/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/eda_mutual_information.png


,feature,mutual_info,mi_normalized
0,avg_word_len,0.08182,0.03166
1,n_chars,0.08093,0.03131
2,std_word_len,0.07993,0.03092
3,punct_ratio,0.07604,0.02942
4,unique_char_ratio,0.07565,0.02927
5,long_word_ratio,0.06180,0.02391
6,upper_ratio,0.05764,0.02230
7,cttr,0.05086,0.01968
8,lexical_entropy,0.05012,0.01939
9,max_word_len,0.04950,0.01915


## 5. Redundancia entre features densas

In [7]:
print(eda.fig_correlation(dense_train, features))
correlation = dense_train[features].corr(method="spearman").abs()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
pairs = (upper.stack().sort_values(ascending=False).head(12).rename("|rho|")
         .reset_index().rename(columns={"level_0": "feature_a", "level_1": "feature_b"}))
print("Pares con mayor redundancia (|rho| de Spearman):")
pairs.round(3)

/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/eda_correlacion_features.png


Pares con mayor redundancia (|rho| de Spearman):


,feature_a,feature_b,|rho|
0,n_emoticon,has_emoticon,1.000
1,ttr,hapax_ratio,0.999
2,n_unique_words,lexical_entropy,0.998
3,stopword_ratio,content_ratio,0.997
4,cttr,lexical_entropy,0.997
5,n_unique_words,cttr,0.991
6,n_words,n_unique_words,0.989
7,n_words,lexical_entropy,0.980
8,punct_ratio,punct_per_word,0.969
9,n_words,cttr,0.963


**INTERPRETACIÓN.** Hay bloques de features casi colineales (longitud, palabras, oraciones;
TTR y CTTR; emoji y `has_emoji`). No se eliminan todavía: la ablación dirá si aportan o solo
duplican. Se documenta la redundancia para interpretar importancias más adelante.

## 6. Distribución de la longitud por clase

In [8]:
print(eda.fig_length_histograms(train))
length_by_class = (train.assign(n=dense_train["n_chars"])
                   .groupby(cfg.TARGET, observed=True)["n"].describe()[["mean", "50%", "std"]])
length_by_class.round(2)

/home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/eda_longitud_densidad.png


,mean,50%,std
age_range,,,
13-17,73.47,66.0,39.60
18-24,72.57,66.0,38.61
25-34,84.77,84.0,40.07
35-49,96.03,103.0,39.56
50-64,104.41,121.0,38.51
65-xx,110.97,131.0,36.57


## 7. Marcadores estilométricos por clase

In [9]:
style_table = dense_train[textometry.STYLE_FEATURES].copy()
style_table["clase"] = np.asarray(y_train)
style_means = style_table.groupby("clase").mean().T.round(4)
style_means.to_csv(cfg.TABLES / "eda_estilo_por_clase.csv")
style_means

clase,13-17,18-24,25-34,35-49,50-64,65-xx
upper_ratio,0.1537,0.1309,0.1231,0.1269,0.1463,0.1533
digit_ratio,0.0188,0.0176,0.0189,0.0206,0.0201,0.0214
punct_ratio,0.0728,0.0697,0.0728,0.0730,0.0676,0.0715
n_excl,0.2198,0.2305,0.3009,0.2549,0.1795,0.1823
n_quest,0.1006,0.1022,0.1017,0.0977,0.0788,0.0706
n_open_punct,0.0010,0.0003,0.0004,0.0001,0.0000,0.0000
multi_punct_count,0.1291,0.1543,0.2063,0.1955,0.1580,0.1239
elongated_count,0.0815,0.0713,0.0618,0.0405,0.0338,0.0262
all_caps_count,0.9066,0.6902,0.6495,0.7528,1.1182,1.4048
has_all_caps,0.4525,0.3828,0.3857,0.4416,0.4785,0.6493


**OBSERVACIÓN.** Los marcadores informales (emojis, elongaciones, mayúsculas, signos de
exclamación) se concentran en `13-17`, mientras que las URLs y las menciones crecen con la
edad. Los retuits son más frecuentes en `65-xx` (≈53%) que en `18-24` (≈30%).

**HIPÓTESIS H2.** El estilo informal (elongaciones, mayúsculas, puntuación repetida, emojis)
discrimina sobre todo las clases extremas (13-17 frente a 65-xx), y será menos útil entre
clases contiguas (35-49 vs 50-64).

**CONSECUENCIA.** Se espera una matriz de confusión con errores concentrados en clases
adyacentes; se medirá en la Fase 17 (error analysis).

## 8. Hallazgos del EDA y features candidatas

In [10]:
top_kruskal = kruskal_table.head(10)[["feature", "epsilon_sq"]]
top_mi = mi_table.head(10)[["feature", "mi_normalized"]]
findings = {
    "n_features_densas": len(features),
    "top_kruskal": top_kruskal.to_dict(orient="records"),
    "top_mi": top_mi.to_dict(orient="records"),
    "redundancia_max": float(np.nanmax(upper.to_numpy())),
    "pares_redundantes": pairs.head(5).to_dict(orient="records"),
}
import json
(cfg.EXPERIMENTS / "eda_02_resumen.json").write_text(
    json.dumps(findings, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(findings, indent=2, ensure_ascii=False)[:1500])

{
  "n_features_densas": 38,
  "top_kruskal": [
    {
      "feature": "n_chars",
      "epsilon_sq": 0.12550992051299326
    },
    {
      "feature": "unique_char_ratio",
      "epsilon_sq": 0.06689337617317184
    },
    {
      "feature": "cttr",
      "epsilon_sq": 0.06641983136020597
    },
    {
      "feature": "lexical_entropy",
      "epsilon_sq": 0.065157820913289
    },
    {
      "feature": "n_unique_words",
      "epsilon_sq": 0.06446749103830333
    },
    {
      "feature": "n_words",
      "epsilon_sq": 0.06054215401250117
    },
    {
      "feature": "n_mention",
      "epsilon_sq": 0.05211643640778002
    },
    {
      "feature": "max_word_len",
      "epsilon_sq": 0.04685036659315994
    },
    {
      "feature": "long_word_ratio",
      "epsilon_sq": 0.03994207930557268
    },
    {
      "feature": "n_sentences",
      "epsilon_sq": 0.03969267162451217
    }
  ],
  "top_mi": [
    {
      "feature": "avg_word_len",
      "mi_normalized": 0.03165689108936283
   

### Conclusiones de la Fase 2

| # | Observación | Interpretación | Hipótesis | Consecuencia |
|---|---|---|---|---|
| E1 | Longitud y riqueza léxica crecen monótonamente con la edad (ε² alto) | señal de madurez de escritura, replicada en la literatura | H1: textometría aporta señal directa | familia `GROUP_D_TEXTOMETRY` en la ablación |
| E2 | Marcas informales concentradas en `13-17`; URLs/menciones crecen con la edad | el registro (formal/informal) es un eje discriminante | H2: el estilo separa mejor las clases extremas | revisar confusión entre clases adyacentes |
| E3 | Retuits más frecuentes en `65-xx` (53%) que en `18-24` (30%) | el comportamiento de retuit es en sí una señal de actividad | H3: `is_rt` y la proporción de RT aportan información | incluir `is_rt` y contadores de RT |
| E4 | Bloques de features casi colineales (longitud/palabras; TTR/CTTR) | redundancia esperable | H4: parte de la ganancia de añadir features es ilusoria | ablación incremental obligatoria antes de conservar features |
| E5 | Textos vacíos (104) y muy cortos | sin señal textual | H5: en esos casos conviene predecir la clase prior | manejo explícito en el pipeline |

**Siguiente:** Fase 3 (análisis lingüístico: términos distintivos, palabras funcionales y
marcadores discursivos) y consolidación del gate A con la lista priorizada de hipótesis.